# Clustering — Implementations

K-Means, DBSCAN and the Gaussian mixture again, beside their library forms. The recurring theme is *initialisation as identity*: hand two implementations the same starting centers and the same iteration budget and Lloyd's loop and EM land on the same fixed point to machine precision — while DBSCAN, with nothing differentiable or even continuous in it, is compared on order-invariant facts instead. Every fixture draws through NumPy generators, and `tol=0.0` everywhere so no lane leaves a loop early.

## 11_kmeans

Assign to the nearest center, move the center to the mean, repeat.

### torch

The scratch lane's K-Means++ init is replayed in NumPy — same generator, same draws — and only Lloyd's loop moves to tensors. **What torch adds:** `torch.cdist` computes the whole point-to-center distance matrix as one batched op, the shape GPUs are built for; the argmin/mean loop is otherwise untouched (there is nothing to differentiate through an argmin).

In [ ]:
import numpy as np
import torch

# hints:
# 1. Keep the init in NumPy, verbatim — the lane hinges on replaying those exact draws.
# 2. torch.cdist(X, centers)**2 replaces the broadcasted (n,1,d)-(1,K,d) subtraction.
# 3. An empty cluster keeps its old center; clone() before updating rows in place.
# 4. tol=0.0 runs to the exact fixed point: member means stop moving at all.


class KMeansScratch:
    """K-Means with the scratch lane's exact NumPy init; Lloyd's loop on tensors."""

    def __init__(self, n_clusters=3, max_iter=100, tol=1e-4,
                 init="kmeans++", random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.init = init
        self.random_state = random_state
        self.cluster_centers_ = None
        self.labels_ = None
        self.inertia_ = None
        self.n_iter_ = 0

    def _init_centers(self, X):
        # NumPy verbatim: the draws must replay the scratch lane bit for bit.
        rng_local = np.random.default_rng(self.random_state)
        n = X.shape[0]
        if self.init == "random":
            idx = rng_local.choice(n, self.n_clusters, replace=False)
            return X[idx].copy()
        centers = [X[rng_local.integers(n)]]
        for _ in range(1, self.n_clusters):
            dists = np.min([np.sum((X - c) ** 2, axis=1) for c in centers], axis=0)
            probs = dists / dists.sum()
            idx = rng_local.choice(n, p=probs)
            centers.append(X[idx])
        return np.array(centers)

    def fit(self, X):
        X_np = np.asarray(X, dtype=float)
        Xt = torch.as_tensor(X_np)
        centers = torch.as_tensor(self._init_centers(X_np)).clone()

        for iteration in range(1, self.max_iter + 1):
            # Assign: one batched distance matrix instead of a broadcast subtract
            sq_dists = torch.cdist(Xt, centers) ** 2
            labels = torch.argmin(sq_dists, dim=1)

            # Update: each center moves to the mean of its members
            new_centers = centers.clone()
            for k in range(self.n_clusters):
                members = labels == k
                if bool(members.any()):
                    new_centers[k] = Xt[members].mean(dim=0)

            shift = float(torch.max(torch.linalg.norm(new_centers - centers, dim=1)))
            centers = new_centers
            self.n_iter_ = iteration
            if shift <= self.tol:
                break

        sq_dists = torch.cdist(Xt, centers) ** 2
        labels = torch.argmin(sq_dists, dim=1)
        self.cluster_centers_ = centers.numpy()
        self.labels_ = labels.numpy()
        self.inertia_ = float(sq_dists[torch.arange(Xt.shape[0]), labels].sum())
        return self

    def predict(self, X):
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        sq_dists = torch.cdist(Xt, torch.as_tensor(self.cluster_centers_)) ** 2
        return torch.argmin(sq_dists, dim=1).numpy()


In [ ]:
# exports: centers, labels_fix, inertia
_rng_fix = np.random.default_rng(11)
_centers_fix = np.array([[2.0, 2.0], [-2.0, -2.0], [2.0, -2.0]])
X_fix = np.vstack([c + _rng_fix.normal(0, 0.5, size=(30, 2)) for c in _centers_fix])

_km_fix = KMeansScratch(n_clusters=3, max_iter=60, tol=0.0,
                        init="kmeans++", random_state=0).fit(X_fix)
centers = _km_fix.cluster_centers_.tolist()
labels_fix = [int(l) for l in _km_fix.labels_]
inertia = float(_km_fix.inertia_)

print("iterations:", _km_fix.n_iter_, " inertia:", round(inertia, 4))


In [ ]:
_c = np.array(centers)
_lab = np.array(labels_fix)
# Lloyd's fixed point: every center is exactly the mean of its members.
for _k in range(3):
    assert np.allclose(_c[_k], X_fix[_lab == _k].mean(axis=0), atol=1e-10), \
        "each center sits at the mean of its cluster"
_sq = np.sum((X_fix[:, None, :] - _c[None, :, :]) ** 2, axis=2)
assert np.array_equal(np.argmin(_sq, axis=1), _lab), "labels are nearest-center assignments"
assert abs(inertia - float(_sq[np.arange(len(X_fix)), _lab].sum())) < 1e-8, \
    "inertia is the summed within-cluster squared distance"


### library

`KMeans(init=<explicit array>, n_init=1, algorithm='lloyd', tol=0.0)` — pinned to the scratch start, it must reach the identical fixed point, cluster numbering included. **What the library adds:** k-means++ with several restarts, Elkan's triangle-inequality speedups, and a `tol` whose semantics (relative, squared) silently differ from the scratch lane's.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# hints:
# 1. init= accepts an explicit (K, d) array; n_init=1 stops sklearn re-drawing it.
# 2. algorithm='lloyd' is the plain assign/update loop the scratch lane implements.
# 3. sklearn's tol is relative to data variance and squared — 0.0 is the faithful value.
# 4. Same init plus full convergence means even the cluster numbering matches.


class KMeansScratch:
    """sklearn's KMeans pinned to the scratch lane's exact starting centers."""

    def __init__(self, n_clusters=3, max_iter=100, tol=1e-4,
                 init="kmeans++", random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.init = init
        self.random_state = random_state

    def _init_centers(self, X):
        # The scratch lane's init, replayed in NumPy so sklearn starts where it did.
        rng_local = np.random.default_rng(self.random_state)
        n = X.shape[0]
        if self.init == "random":
            idx = rng_local.choice(n, self.n_clusters, replace=False)
            return X[idx].copy()
        centers = [X[rng_local.integers(n)]]
        for _ in range(1, self.n_clusters):
            dists = np.min([np.sum((X - c) ** 2, axis=1) for c in centers], axis=0)
            probs = dists / dists.sum()
            idx = rng_local.choice(n, p=probs)
            centers.append(X[idx])
        return np.array(centers)

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        # sklearn's tol compares *squared* center shifts *relative* to the data
        # variance — a different quantity from the scratch lane's absolute shift.
        # 0.0 is the only setting that means the same thing in both: run until
        # the assignments stop changing.
        self._model = KMeans(n_clusters=self.n_clusters, init=self._init_centers(X),
                             n_init=1, algorithm="lloyd", max_iter=self.max_iter,
                             tol=0.0).fit(X)
        self.cluster_centers_ = self._model.cluster_centers_
        self.labels_ = self._model.labels_
        self.inertia_ = float(self._model.inertia_)
        self.n_iter_ = int(self._model.n_iter_)
        return self

    def predict(self, X):
        return self._model.predict(np.asarray(X, dtype=float))


In [ ]:
# exports: centers, labels_fix, inertia
_rng_fix = np.random.default_rng(11)
_centers_fix = np.array([[2.0, 2.0], [-2.0, -2.0], [2.0, -2.0]])
X_fix = np.vstack([c + _rng_fix.normal(0, 0.5, size=(30, 2)) for c in _centers_fix])

_km_fix = KMeansScratch(n_clusters=3, max_iter=60, tol=0.0,
                        init="kmeans++", random_state=0).fit(X_fix)
centers = _km_fix.cluster_centers_.tolist()
labels_fix = [int(l) for l in _km_fix.labels_]
inertia = float(_km_fix.inertia_)

print("iterations:", _km_fix.n_iter_, " inertia:", round(inertia, 4))


In [ ]:
_c = np.array(centers)
_lab = np.array(labels_fix)
for _k in range(3):
    assert np.allclose(_c[_k], X_fix[_lab == _k].mean(axis=0), atol=1e-10), \
        "sklearn converged to the same Lloyd fixed point"
assert np.array_equal(_km_fix.predict(X_fix), _lab), "predict reproduces the training labels"
assert _km_fix.n_iter_ <= 60, "Lloyd converged inside the shared iteration budget"


## 11_dbscan

Clusters are dense regions: core points within eps of enough neighbours, expanded transitively. **No torch lane:** density reachability is a discrete graph traversal — neighbour sets, a BFS queue, integer labels — with no continuous quantity to differentiate, so a tensor version would be the NumPy code with different spelling.

### library

sklearn's `DBSCAN` under the scratch attribute names. Cluster *numbers* depend on visit order, so the honest comparison is order-invariant: how many clusters, how many noise points, which sizes. Both sides count the point itself toward `min_samples`. **What the library adds:** a KD-tree/ball-tree neighbour index in place of the O(n²) distance matrix, and `core_sample_indices_` exposed for inspection.

In [ ]:
import numpy as np
from sklearn.cluster import DBSCAN

# hints:
# 1. sklearn counts the point itself toward min_samples — so does the scratch lane.
# 2. Label numbers depend on visit order; compare counts and sizes, not raw labels.
# 3. core_sample_indices_ is order-invariant: the same set for any traversal.
# 4. Noise (-1) is what remains outside every core point's eps-ball.


class DBSCANScratch:
    """sklearn's DBSCAN behind the scratch lane's attribute names."""

    def __init__(self, eps=0.5, min_samples=5):
        self.eps = eps
        self.min_samples = min_samples
        self.labels_ = None

    def fit(self, X):
        self._model = DBSCAN(eps=self.eps, min_samples=self.min_samples)
        self._model.fit(np.asarray(X, dtype=float))
        self.labels_ = self._model.labels_
        return self


In [ ]:
# exports: n_clusters, n_noise, cluster_sizes
_rng_fix = np.random.default_rng(3)
_centers_fix = np.array([[2.0, 2.0], [-2.0, -2.0], [2.0, -2.0]])
X_fix = np.vstack([c + _rng_fix.normal(0, 0.35, size=(30, 2)) for c in _centers_fix]
                  + [_rng_fix.uniform(-5.5, 5.5, size=(8, 2))])

_db_fix = DBSCANScratch(eps=0.7, min_samples=5).fit(X_fix)
n_clusters = int(len(set(_db_fix.labels_) - {-1}))
n_noise = int(np.sum(_db_fix.labels_ == -1))
cluster_sizes = sorted(int(np.sum(_db_fix.labels_ == k)) for k in range(n_clusters))

print("clusters:", n_clusters, " noise:", n_noise, " sizes:", cluster_sizes)


In [ ]:
# Core points are order-invariant: recompute them by brute force, with the
# point itself counted inside its own eps-ball, exactly as the scratch lane does.
_D = np.sqrt(np.sum((X_fix[:, None, :] - X_fix[None, :, :]) ** 2, axis=2))
_core = np.sum(_D <= _db_fix.eps, axis=1) >= _db_fix.min_samples
assert set(np.where(_core)[0]) == set(_db_fix._model.core_sample_indices_), \
    "sklearn agrees on which points are core"
for _i in np.where(_db_fix.labels_ == -1)[0]:
    assert not np.any(_core & (_D[_i] <= _db_fix.eps)), \
        "noise has no core point within eps — unreachable by definition"
assert set(_db_fix.labels_) - {-1} == set(range(n_clusters)), \
    "cluster ids are contiguous from 0"
assert sum(cluster_sizes) + n_noise == len(X_fix), "every point is a member or noise"


## 11_gmm_em

Soft K-Means with covariances: E-step responsibilities, M-step re-estimation, likelihood never decreasing.

### torch

The same EM, tensorised, from the same start — the scratch lane's own K-Means init is replayed verbatim, then E and M mirror the NumPy code line by line at float64. **What torch adds:** an audit EM cannot do for itself — autograd differentiates the log-likelihood at the fitted means and finds the gradient ~0, confirming EM reached a stationary point without ever computing a gradient.

In [ ]:
import math

import numpy as np
import torch

# hints:
# 1. The scratch model initialises from its own K-Means — replay that init exactly.
# 2. Mirror E then M per iteration; labels_ reads the resp of the LAST E-step.
# 3. Everything float64; torch.linalg.inv/det on each (d, d) covariance slice.
# 4. tol=0.0 plus a fixed max_iter keeps every lane on the same iteration count.
# 5. Autograd can audit the result: at the fitted means, grad of the ll is ~0.


def _kmeans_centers(X, K, seed):
    """Replay KMeansScratch(n_clusters=K, random_state=seed).fit(X) verbatim:
    same K-Means++ draws from the same NumPy generator, same Lloyd loop, same
    stopping rule — the returned centers match the scratch lane bit for bit."""
    rng_local = np.random.default_rng(seed)
    n = X.shape[0]
    centers = [X[rng_local.integers(n)]]
    for _ in range(1, K):
        dists = np.min([np.sum((X - c) ** 2, axis=1) for c in centers], axis=0)
        idx = rng_local.choice(n, p=dists / dists.sum())
        centers.append(X[idx])
    centers = np.array(centers)
    for _ in range(100):
        sq = np.sum((X[:, None, :] - centers[None, :, :]) ** 2, axis=2)
        labels = np.argmin(sq, axis=1)
        new_centers = np.array([X[labels == k].mean(axis=0) if np.any(labels == k)
                                else centers[k] for k in range(K)])
        shift = np.max(np.linalg.norm(new_centers - centers, axis=1))
        centers = new_centers
        if shift <= 1e-4:
            break
    return centers


class GMMScratch:
    """Gaussian mixture by EM on tensors, mirroring the scratch loop line by line."""

    def __init__(self, n_components=3, max_iter=100, tol=1e-6,
                 reg_covar=1e-6, random_state=42):
        self.n_components = n_components
        self.max_iter = max_iter
        self.tol = tol
        self.reg_covar = reg_covar
        self.random_state = random_state

    @staticmethod
    def _multivariate_gaussian(X, mu, sigma):
        """Evaluate N(x; mu, sigma) for each row of the tensor X."""
        d = X.shape[1]
        diff = X - mu
        sigma_inv = torch.linalg.inv(sigma)
        det_sigma = torch.linalg.det(sigma)
        maha = torch.sum(diff @ sigma_inv * diff, dim=1)
        norm_const = torch.sqrt((2 * math.pi) ** d * det_sigma)
        return torch.exp(-0.5 * maha) / norm_const

    def fit(self, X):
        X_np = np.asarray(X, dtype=float)
        n, d = X_np.shape
        K = self.n_components
        Xt = torch.as_tensor(X_np)

        # Same start as the scratch lane: its own K-Means means, unit
        # covariances, uniform weights.
        means = torch.as_tensor(_kmeans_centers(X_np, K, self.random_state)).clone()
        covs = torch.stack([torch.eye(d, dtype=torch.float64) for _ in range(K)])
        weights = torch.full((K,), 1.0 / K, dtype=torch.float64)

        self.log_likelihoods_ = []
        for iteration in range(self.max_iter):
            # E-step: responsibilities under the current parameters
            resp = torch.zeros((n, K), dtype=torch.float64)
            for k in range(K):
                resp[:, k] = weights[k] * self._multivariate_gaussian(Xt, means[k], covs[k])
            resp_sum = torch.clamp(resp.sum(dim=1, keepdim=True), min=1e-300)
            resp = resp / resp_sum

            self.log_likelihoods_.append(float(torch.log(resp_sum.ravel()).sum()))
            if len(self.log_likelihoods_) > 1:
                if abs(self.log_likelihoods_[-1] - self.log_likelihoods_[-2]) < self.tol:
                    break

            # M-step: re-estimate weights, means, covariances from resp
            Nk = resp.sum(dim=0)
            for k in range(K):
                means[k] = (resp[:, k] @ Xt) / Nk[k]
                diff = Xt - means[k]
                covs[k] = ((resp[:, k:k + 1] * diff).T @ diff / Nk[k]
                           + self.reg_covar * torch.eye(d, dtype=torch.float64))
                weights[k] = Nk[k] / n

        self.means_ = means.numpy()
        self.covariances_ = covs.numpy()
        self.weights_ = weights.numpy()
        self.responsibilities_ = resp.numpy()
        self.labels_ = torch.argmax(resp, dim=1).numpy()
        self.n_iter_ = iteration + 1
        return self

    def predict(self, X):
        Xt = torch.as_tensor(np.asarray(X, dtype=float))
        K = self.n_components
        resp = torch.zeros((Xt.shape[0], K), dtype=torch.float64)
        for k in range(K):
            resp[:, k] = float(self.weights_[k]) * self._multivariate_gaussian(
                Xt, torch.as_tensor(self.means_[k]), torch.as_tensor(self.covariances_[k]))
        return torch.argmax(resp, dim=1).numpy()


In [ ]:
# exports: means, weights, covariances, labels_fix
_rng_fix = np.random.default_rng(7)
_centers_fix = np.array([[2.0, 2.0], [-2.0, -2.0], [2.0, -2.0]])
X_fix = np.vstack([c + _rng_fix.normal(0, 0.5, size=(30, 2)) for c in _centers_fix])

_gmm_fix = GMMScratch(n_components=3, max_iter=25, tol=0.0,
                      reg_covar=1e-6, random_state=0).fit(X_fix)
means = _gmm_fix.means_.tolist()
weights = _gmm_fix.weights_.tolist()
covariances = _gmm_fix.covariances_.tolist()
labels_fix = [int(l) for l in _gmm_fix.labels_]

print("weights:", np.round(np.array(weights), 4))
print("final ll:", round(_gmm_fix.log_likelihoods_[-1], 4))


In [ ]:
_ll = np.array(_gmm_fix.log_likelihoods_)
assert np.all(np.diff(_ll) >= -1e-9), "EM never decreases the log-likelihood"
assert abs(sum(weights) - 1.0) < 1e-12, "mixture weights stay a distribution"

# What autograd is for here: at the fitted parameters the total log-likelihood
# should be stationary in the means — EM found a (local) maximum.
_Xt = torch.as_tensor(X_fix)
_mu = torch.as_tensor(_gmm_fix.means_).clone().requires_grad_(True)
_dens = []
for _k in range(3):
    _S = torch.as_tensor(_gmm_fix.covariances_[_k])
    _diff = _Xt - _mu[_k]
    _maha = torch.sum(_diff @ torch.linalg.inv(_S) * _diff, dim=1)
    _dens.append(float(_gmm_fix.weights_[_k]) * torch.exp(-0.5 * _maha)
                 / torch.sqrt((2 * math.pi) ** _Xt.shape[1] * torch.linalg.det(_S)))
_ll_total = torch.log(torch.stack(_dens, dim=1).sum(dim=1)).sum()
_ll_total.backward()
assert float(_mu.grad.abs().max()) < 1e-6, "fitted means are a stationary point of the ll"


### library

`GaussianMixture` accepts the full starting point — `means_init`, `weights_init`, `precisions_init` — and its default `reg_covar=1e-6` is exactly the scratch regulariser, so with `tol=0.0` and the same `max_iter` the runs are step-for-step identical. **What the library adds:** log-domain E-steps via Cholesky factors (the numerically safe route), yet the final parameters agree to ~1e-15.

In [ ]:
import warnings

import numpy as np
from sklearn.mixture import GaussianMixture

# hints:
# 1. means_init/weights_init/precisions_init pin sklearn to the scratch start point.
# 2. precisions_init wants inverse covariances — identity is its own inverse.
# 3. reg_covar=1e-6 is sklearn's default AND the scratch regulariser: keep them equal.
# 4. tol=0.0 disables early exit; the ConvergenceWarning it triggers is expected.


def _kmeans_centers(X, K, seed):
    """Replay KMeansScratch(n_clusters=K, random_state=seed).fit(X) verbatim:
    same K-Means++ draws from the same NumPy generator, same Lloyd loop, same
    stopping rule — the returned centers match the scratch lane bit for bit."""
    rng_local = np.random.default_rng(seed)
    n = X.shape[0]
    centers = [X[rng_local.integers(n)]]
    for _ in range(1, K):
        dists = np.min([np.sum((X - c) ** 2, axis=1) for c in centers], axis=0)
        idx = rng_local.choice(n, p=dists / dists.sum())
        centers.append(X[idx])
    centers = np.array(centers)
    for _ in range(100):
        sq = np.sum((X[:, None, :] - centers[None, :, :]) ** 2, axis=2)
        labels = np.argmin(sq, axis=1)
        new_centers = np.array([X[labels == k].mean(axis=0) if np.any(labels == k)
                                else centers[k] for k in range(K)])
        shift = np.max(np.linalg.norm(new_centers - centers, axis=1))
        centers = new_centers
        if shift <= 1e-4:
            break
    return centers


class GMMScratch:
    """sklearn's GaussianMixture initialised exactly where the scratch lane starts."""

    def __init__(self, n_components=3, max_iter=100, tol=1e-6,
                 reg_covar=1e-6, random_state=42):
        self.n_components = n_components
        self.max_iter = max_iter
        self.tol = tol
        self.reg_covar = reg_covar
        self.random_state = random_state

    def fit(self, X):
        X = np.asarray(X, dtype=float)
        K, d = self.n_components, X.shape[1]
        self._model = GaussianMixture(
            n_components=K, covariance_type="full", n_init=1,
            max_iter=self.max_iter, tol=self.tol, reg_covar=self.reg_covar,
            weights_init=np.ones(K) / K,
            means_init=_kmeans_centers(X, K, self.random_state),
            precisions_init=np.array([np.eye(d) for _ in range(K)]))
        with warnings.catch_warnings():
            # tol=0.0 never satisfies sklearn's convergence test — by design,
            # so no lane leaves the loop before the shared max_iter.
            warnings.simplefilter("ignore")
            self._model.fit(X)
        self.means_ = self._model.means_
        self.covariances_ = self._model.covariances_
        self.weights_ = self._model.weights_
        self.labels_ = self._model.predict(X)
        # sklearn's lower_bound_ is the last E-step's *mean* log-likelihood —
        # the scratch lane's log_likelihoods_[-1], divided by n.
        self.log_likelihoods_ = [float(self._model.lower_bound_) * len(X)]
        return self

    def predict(self, X):
        return self._model.predict(np.asarray(X, dtype=float))


In [ ]:
# exports: means, weights, covariances, labels_fix
_rng_fix = np.random.default_rng(7)
_centers_fix = np.array([[2.0, 2.0], [-2.0, -2.0], [2.0, -2.0]])
X_fix = np.vstack([c + _rng_fix.normal(0, 0.5, size=(30, 2)) for c in _centers_fix])

_gmm_fix = GMMScratch(n_components=3, max_iter=25, tol=0.0,
                      reg_covar=1e-6, random_state=0).fit(X_fix)
means = _gmm_fix.means_.tolist()
weights = _gmm_fix.weights_.tolist()
covariances = _gmm_fix.covariances_.tolist()
labels_fix = [int(l) for l in _gmm_fix.labels_]

print("weights:", np.round(np.array(weights), 4))
print("final ll:", round(_gmm_fix.log_likelihoods_[-1], 4))


In [ ]:
assert abs(sum(weights) - 1.0) < 1e-12, "mixture weights stay a distribution"
_proba = _gmm_fix._model.predict_proba(X_fix)
assert np.max(np.abs(_proba.sum(axis=1) - 1.0)) < 1e-9, "responsibilities sum to one per point"
for _S in _gmm_fix.covariances_:
    assert np.all(np.linalg.eigvalsh(_S) > 0), "reg_covar keeps every covariance positive-definite"
assert np.array_equal(np.argmax(_proba, axis=1), np.array(labels_fix)), \
    "hard labels are the argmax responsibility"
